# Multi-Output ANN + GradNorm (190 features) — Results Viewer

Displays results from the GradNorm multi-output model trained by `run_training_gradnorm.py`.
No training is performed here.

**What changed from the homoscedastic model**:
- Input: 171 → **190 features** (171 colors + 19 absolute magnitudes)
- Loss: HomoscedasticUncertaintyLoss → **GradNorm** (α=1.5)
  - GradNorm prevents the 194:1 task weight collapse seen in the previous run
  - Task weights should stay near 1:1 (both tasks sum to N_TASKS=2)

**Baselines**:
| Model | log g R² | log g RMSE |
|---|---|---|
| Residual ANN, single-output | 0.519 | 0.199 dex |
| Multi-output (homoscedastic) | 0.483 | 0.207 dex |

**Expected artifacts**:
- `models/multi_output_gradnorm/stellar_multi_output_gradnorm_best.pth`
- `models/multi_output_gradnorm/scaler_gradnorm.pkl`
- `results/multi_output_gradnorm/test_metrics.json`

## 1. Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import joblib
import warnings
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from IPython.display import Image as IPyImage, display as ipy_display

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

warnings.filterwarnings('ignore')
sns.set_context('talk')

DEVICE = torch.device('cpu')
SEED   = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
print(f'Device: {DEVICE}')

## 2. Paths

In [ ]:
PROJECT_ROOT = Path('C:/git_repo/cool-dwarf_stellar_parameter_inference_from_survey_data')
DATA_PATH    = PROJECT_ROOT / 'data' / 'logg_final_df' / 'cool_dwarf_catalog_FGKM_consolidated.csv'
MODELS_DIR   = PROJECT_ROOT / 'models'  / 'multi_output_gradnorm'
RESULTS_DIR  = PROJECT_ROOT / 'results' / 'multi_output_gradnorm'
MODEL_PATH   = MODELS_DIR / 'stellar_multi_output_gradnorm_best.pth'
SCALER_PATH  = MODELS_DIR / 'scaler_gradnorm.pkl'

for p in [MODEL_PATH, SCALER_PATH, RESULTS_DIR / 'test_metrics.json']:
    print(f'  [{"OK" if p.exists() else "MISSING"}] {p.name}')

## 3. Data Loading & Feature Engineering (190 features)

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f'Dataset: {df.shape}')

sorted_mags = [
    'A_BAP', 'A_GSD', 'A_ps_g', 'A_BP', 'A_VAP', 'A_ps_r', 'A_RSD', 'A_RAP',
    'A_GG', 'A_ps_i', 'A_ISD', 'A_RP', 'A_ps_z', 'A_ps_y', 'A_J', 'A_H',
    'A_KS', 'A_W1', 'A_W2'
]
COLOR_COLS   = [f'COLOR_{sorted_mags[i]}_{sorted_mags[j]}'
                for i in range(len(sorted_mags)) for j in range(i+1, len(sorted_mags))]
ABS_MAG_COLS = [b.replace('A_', 'M_') for b in sorted_mags]

dist_pc     = df['distance_gaia_pc'].values.astype(np.float64)
dist_mod    = 5.0 * np.log10(dist_pc / 10.0)
abs_mag_arr = np.column_stack([df[b].values - dist_mod for b in sorted_mags]).astype(np.float32)

FEATURE_COLS = COLOR_COLS + ABS_MAG_COLS
X            = np.hstack([df[COLOR_COLS].values.astype(np.float32), abs_mag_arr])

log10_teff     = np.log10(df['teff'].values).astype(np.float32)
logg           = df['logg'].values.astype(np.float32)
y              = np.column_stack([log10_teff, logg]).astype(np.float32)
spectral_types = df['spectral_type_group'].values

print(f'Features: {X.shape[1]}  |  Targets: {y.shape}')

## 4. Train / Val / Test Split

In [ ]:
_, X_temp, _, y_temp, _, st_temp = train_test_split(
    X, y, spectral_types, test_size=0.30, random_state=SEED, stratify=spectral_types)
_, X_test, _, y_test, _, st_test = train_test_split(
    X_temp, y_temp, st_temp, test_size=0.50, random_state=SEED, stratify=st_temp)

print(f'Test set: {X_test.shape[0]:,} samples')

## 5. Load Saved Scaler & DataLoader

In [ ]:
scaler        = joblib.load(SCALER_PATH)
X_test_scaled = scaler.transform(X_test).astype(np.float32)

class StellarMultiDataset(Dataset):
    def __init__(self, features, targets):
        self.X = torch.tensor(features, dtype=torch.float32)
        self.y = torch.tensor(targets,  dtype=torch.float32)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

test_loader = DataLoader(StellarMultiDataset(X_test_scaled, y_test),
                         batch_size=2048, shuffle=False, num_workers=0)
print(f'Scaler loaded. Test DataLoader: {len(test_loader)} batches')

## 6. Model Architecture

```
Input (190)
    └── ResBlock(190→256)
            └── ResBlock(256→128)
                    ├── Teff head: Linear(128→64) → BN → ReLU → Dropout → Linear(1)
                    └── log g head: Linear(128→64) → BN → ReLU → Dropout → Linear(1)
```

Same architecture as the homoscedastic model — the only difference is the loss function.

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, in_dim, out_dim, dropout=0.15):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(in_dim, out_dim), nn.BatchNorm1d(out_dim), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(out_dim, out_dim), nn.BatchNorm1d(out_dim),
        )
        self.skip = nn.Linear(in_dim, out_dim) if in_dim != out_dim else nn.Identity()
        self.relu = nn.ReLU()
    def forward(self, x): return self.relu(self.block(x) + self.skip(x))

class StellarMultiOutputNet(nn.Module):
    def __init__(self, input_dim, dropout=0.15):
        super().__init__()
        self.backbone = nn.Sequential(
            ResBlock(input_dim, 256, dropout),
            ResBlock(256, 128, dropout),
        )
        self.teff_head = nn.Sequential(
            nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.10), nn.Linear(64, 1))
        self.logg_head = nn.Sequential(
            nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.10), nn.Linear(64, 1))
    def forward(self, x):
        shared = self.backbone(x)
        return torch.cat([self.teff_head(shared), self.logg_head(shared)], dim=1)

model = StellarMultiOutputNet(input_dim=len(FEATURE_COLS)).to(DEVICE)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

## 7. Load Saved Model

In [ ]:
checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f'Loaded model from: {MODEL_PATH}')
print(f'  Best epoch:    {checkpoint["best_epoch"]}')
print(f'  Best val loss: {checkpoint["best_val_loss"]:.6f}')
print(f'  GradNorm α:    {checkpoint.get("gradnorm_alpha", "N/A")}')

task_weights = checkpoint['task_weights']
print(f'\nFinal GradNorm task weights (normalized to sum=2):')
print(f'  Teff  weight = {float(task_weights[0]):.4f}')
print(f'  log g weight = {float(task_weights[1]):.4f}')
print(f'  Ratio = {float(task_weights[0])/float(task_weights[1]):.2f}:1  (vs 194:1 in homoscedastic run)')

## 8. Training Diagnostics

Shows: combined loss, per-task MSE, GradNorm task weights history, LR schedule.

In [ ]:
diag = RESULTS_DIR / 'training_diagnostics.png'
if diag.exists():
    ipy_display(IPyImage(filename=str(diag), width=1400))
else:
    print(f'Not found: {diag}')

## 9. Test Set Evaluation

In [ ]:
model.eval()
all_preds, all_targets = [], []
with torch.no_grad():
    for X_b, y_b in test_loader:
        all_preds.append(model(X_b.to(DEVICE)).cpu().numpy())
        all_targets.append(y_b.numpy())

preds_arr   = np.concatenate(all_preds)
targets_arr = np.concatenate(all_targets)

log10_teff_pred = preds_arr[:, 0];  log10_teff_true = targets_arr[:, 0]
teff_pred_K     = 10.0 ** log10_teff_pred;  teff_true_K = 10.0 ** log10_teff_true
logg_pred       = preds_arr[:, 1];  logg_true = targets_arr[:, 1]

rmse_teff     = np.sqrt(mean_squared_error(teff_true_K, teff_pred_K))
mae_teff      = mean_absolute_error(teff_true_K, teff_pred_K)
r2_teff       = r2_score(teff_true_K, teff_pred_K)
r2_log10_teff = r2_score(log10_teff_true, log10_teff_pred)
rmse_logg     = np.sqrt(mean_squared_error(logg_true, logg_pred))
mae_logg      = mean_absolute_error(logg_true, logg_pred)
r2_logg       = r2_score(logg_true, logg_pred)

print('=' * 60)
print('  TEST SET — Multi-Output ANN + GradNorm')
print('=' * 60)
print(f'  Teff:  RMSE={rmse_teff:.2f} K   MAE={mae_teff:.2f} K   R²={r2_teff:.5f}  R²(log10)={r2_log10_teff:.5f}')
print(f'  log g: RMSE={rmse_logg:.4f} dex  MAE={mae_logg:.4f} dex  R²={r2_logg:.5f}')
print()
print(f'  {"Type":<4} {"N":>8}  {"Teff R²":>9}  {"log g R²":>10}  {"logg RMSE":>10}')
for stype in ["F","G","K","M"]:
    mask = st_test == stype
    if not mask.any(): continue
    tr2   = r2_score(teff_true_K[mask], teff_pred_K[mask])
    lr2   = r2_score(logg_true[mask], logg_pred[mask])
    lrmse = np.sqrt(mean_squared_error(logg_true[mask], logg_pred[mask]))
    print(f'  {stype:<4} {mask.sum():>8,}  {tr2:>9.4f}  {lr2:>10.4f}  {lrmse:>10.4f}')

## 10. One-to-One Plots

In [ ]:
oto = RESULTS_DIR / 'one_to_one_plots.png'
if oto.exists():
    ipy_display(IPyImage(filename=str(oto), width=1400))
else:
    fig, axes = plt.subplots(1,2,figsize=(18,8))
    for ax, true_v, pred_v, xlabel, title, fmt in [
        (axes[0], teff_true_K, teff_pred_K, 'True Teff (K)', 'Teff', f'RMSE={rmse_teff:.1f} K\nR²={r2_teff:.5f}'),
        (axes[1], logg_true,   logg_pred,   'True log g (dex)', 'log g', f'RMSE={rmse_logg:.4f} dex\nR²={r2_logg:.5f}'),
    ]:
        hb = ax.hexbin(true_v, pred_v, gridsize=200, cmap='inferno', mincnt=1, bins='log')
        plt.colorbar(hb, ax=ax, label='log10(count)')
        pad = 100 if 'Teff' in xlabel else 0.1
        lims = [min(true_v.min(),pred_v.min())-pad, max(true_v.max(),pred_v.max())+pad]
        ax.plot(lims, lims, 'r--', lw=2, alpha=0.8)
        ax.set_xlim(lims); ax.set_ylim(lims); ax.set_aspect('equal')
        ax.set_xlabel(xlabel); ax.set_title(f'{title} — GradNorm ANN')
        ax.text(0.97, 0.03, fmt, transform=ax.transAxes, fontsize=11, va='bottom', ha='right',
                bbox=dict(boxstyle='round,pad=0.4', fc='white', alpha=0.8))
    plt.tight_layout(); plt.show()

## 11. Residual Plots

In [ ]:
rp = RESULTS_DIR / 'residual_plots.png'
if rp.exists():
    ipy_display(IPyImage(filename=str(rp), width=1400))
else:
    colors_map = {'F':'#1f77b4','G':'#2ca02c','K':'#ff7f0e','M':'#d62728'}
    fig, axes = plt.subplots(2,2,figsize=(20,12))
    for row, (res, true_v, ylabel, title) in enumerate([
        (teff_pred_K - teff_true_K, teff_true_K, 'Residual (K)',   'Teff'),
        (logg_pred - logg_true,     logg_true,   'Residual (dex)', 'log g'),
    ]):
        ax = axes[row, 0]
        for stype in ['F','G','K','M']:
            mask = st_test == stype
            ax.scatter(true_v[mask], res[mask], alpha=0.05, s=1, color=colors_map[stype], label=stype, rasterized=True)
        ax.axhline(0, color='black', lw=1)
        ax.set_ylabel(ylabel); ax.set_title(f'{title} Residuals'); ax.legend(markerscale=20); ax.grid(True, alpha=0.3)
        ax = axes[row, 1]
        ax.hist(res, bins=200, alpha=0.7, color='steelblue', edgecolor='none')
        ax.axvline(0, color='red', ls='--', lw=1.5)
        ax.axvline(np.mean(res), color='orange', ls='--', lw=1.5, label=f'Mean={np.mean(res):.4g}')
        ax.set_title(f'{title} Residual Distribution'); ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

## 12. Saved Metrics

In [ ]:
with open(RESULTS_DIR / 'test_metrics.json') as f:
    metrics = json.load(f)
print(json.dumps(metrics, indent=2))

## 13. Comparison with Baselines

In [ ]:
print('=' * 68)
print('  COMPARISON — log g')
print('=' * 68)
print(f'  {"Residual ANN, single-output":<40}  R²=0.51924  RMSE=0.19940 dex')
print(f'  {"Multi-output (homoscedastic, 171 feat)":<40}  R²=0.48343  RMSE=0.20674 dex')
flag = 'IMPROVED' if r2_logg > 0.51924 else 'REGRESSED'
print(f'  {"Multi-output + GradNorm (this)":<40}  R²={r2_logg:.5f}  RMSE={rmse_logg:.5f} dex'
      f'  ({r2_logg-0.51924:+.5f} R²)  {flag}')
print()
print('  COMPARISON — Teff')
print('=' * 68)
print(f'  {"4-layer ANN, 171 features (baseline)":<40}  R²=0.96500  RMSE=144.0 K')
print(f'  {"Multi-output + GradNorm (this)":<40}  R²={r2_teff:.5f}  RMSE={rmse_teff:.1f} K')